# 🚢 Titanic 분류 실습 전체 흐름

## 0. 전체 흐름 한눈에 보기

```text
데이터 불러오기
    ↓
데이터 구조 확인
    ↓
1차 EDA
    ↓
1차 전처리 & 파생변수 생성
    ↓
기본 모델 학습 + 1차 점수
    ↓
2차 EDA
    ↓
추가 전처리 & 파생변수 생성
    ↓
인코딩 추가
    ↓
2차/3차 점수 비교
    ↓
Confusion Matrix로 오답 분석
    ↓
교차검증(CV)
    ↓
GridSearchCV로 하이퍼파라미터 튜닝
    ↓
앙상블(Voting)
    ↓
임계치(Threshold) 조정
```

> **핵심:**
> `데이터 확인 → 전처리 → 모델 학습 → 평가 → 오답 분석 → 개선 → 튜닝`
>
> 즉, **한 번 모델을 만들고 끝나는 것이 아니라 점수를 확인하면서 데이터를 계속 개선하는 과정**이다.

---

# 1. Data Load

## 데이터 불러오기

* `train.csv` : 학습 데이터
* `test.csv` : 실제 예측 데이터
* `gender_submission.csv` : Kaggle 제출 형식 확인용

```python
train = pd.read_csv("../../Datas/#titanic/train.csv")
test = pd.read_csv("../../Datas/#titanic/test.csv")
sub = pd.read_csv("../../Datas/#titanic/gender_submission.csv")
```

### 기본 데이터 확인

```python
train.head()
train.info()
train.isna().sum()
```

### 주요 컬럼

| 컬럼         | 의미                 |
| ---------- | ------------------ |
| `survived` | 생존 여부 → **Target** |
| `pclass`   | 객실 등급              |
| `sex`      | 성별                 |
| `age`      | 나이                 |
| `sibsp`    | 형제/배우자 수           |
| `parch`    | 부모/자녀 수            |
| `ticket`   | 티켓 번호              |
| `fare`     | 요금                 |
| `cabin`    | 객실 번호              |
| `embarked` | 탑승 항구              |
| `name`     | 승객 이름              |

### 전처리 전 체크할 것

* 결측치가 있는가?
* 문자열 데이터가 있는가?
* 필요 없는 컬럼이 있는가?
* 파생변수로 만들 수 있는 컬럼이 있는가?

> ⭐ **분류 문제에서는 `survived`가 정답(y), 나머지가 입력 데이터(X)**

---

# 2. 1차 EDA

## EDA란?

> **Exploratory Data Analysis = 탐색적 데이터 분석**

모델을 만들기 전에 **데이터의 특징과 문제점을 파악하는 과정**이다.

### 확인한 주요 내용

* `name` → 이름에서 **호칭 추출 가능**
* `sex` → 성별
* `age` → 결측치 존재
* `sibsp + parch` → 가족 수 파생 가능
* `fare` → 1인당 요금으로 가공 가능
* `cabin` → 결측이 많음
* `embarked` → 결측 처리 및 인코딩 필요

### 데이터 분포 확인

```python
train.hist(figsize=(12, 8))
plt.tight_layout()
plt.show()
```

> ⭐ EDA의 목적은 **"어떤 데이터를 어떻게 가공해야 모델이 잘 학습할까?"**를 찾는 것.

---

# 3. 1차 전처리 & 가공

## 3-1. 범주형 데이터 인코딩

머신러닝 모델은 문자열을 그대로 처리하기 어렵기 때문에
**문자 → 숫자**로 변환한다.

### `sex`

```text
male → 0
female → 1
```

### `embarked`

```text
S → 0
C → 1
Q → 2
```

---

## 3-2. 인코딩 방법

### 방법 ① `map()`

```python
train['sex'] = train['sex'].map({
    "male": 0,
    "female": 1
})
```

### 방법 ② `replace()`

```python
train['sex'] = train['sex'].replace(
    ["male", "female"],
    [0, 1]
)
```

### 방법 ③ `apply(lambda)`

```python
train['sex'] = train['sex'].apply(
    lambda s: 0 if s == 'male' else 1
)
```

### 방법 ④ `LabelEncoder()`

* 범주형 값을 숫자로 변환
* 기본적으로 알파벳 순서 기준으로 숫자 부여

```python
le = LabelEncoder()
train['sex'] = le.fit_transform(train['sex'])
```

### 방법 ⑤ `OneHotEncoder()`

하나의 범주를 하나의 숫자로 표현하지 않고
**각 범주를 별도의 컬럼으로 분리**

```text
sex_male   sex_female
   1           0
   0           1
```

### 방법 ⑥ `pd.get_dummies()`

실무에서 간단하게 원핫인코딩할 때 자주 사용

```python
train = pd.get_dummies(
    train,
    columns=['sex'],
    dtype=int
)
```

> ⭐ **One-Hot Encoding의 핵심 이유**
>
> 범주형 데이터에 **서열 관계가 없는데 숫자를 부여하면 모델이 숫자의 크기를 의미 있는 값으로 오해할 수 있기 때문.**

---

# 3-3. 이름에서 호칭 추출

Titanic 데이터의 `name`에서

```text
"Braund, Mr. Owen Harris"
```

→ `Mr`

처럼 호칭을 추출한다.

```python
train['name'] = train['name'].str.extract(
    r',\s([^\.]+)\.'
)
```

### 호칭 정리

희귀한 호칭들을 대표 호칭으로 통합한다.

```text
Mr
├─ Mr
├─ Sir
├─ Capt
├─ Dr
├─ Rev
└─ ...

Mrs
├─ Mrs
├─ Mme
└─ Dona

Miss
├─ Miss
├─ Mlle
├─ Ms
└─ Lady

Master
└─ Master
```

이후 숫자로 인코딩한다.

```text
Mr     → 0
Miss   → 1
Mrs    → 2
Master → 3
```

> ⭐ **핵심 포인트:**
> `name` 자체보다 **호칭(title)**이라는 새로운 정보를 추출해서 모델에 활용한다.

---

# 3-4. 결측치 처리

## Age

단순히 전체 평균으로 채우는 것이 아니라
**호칭별 평균 나이**를 사용한다.

```python
train['age'] = train['age'].fillna(
    np.round(
        train.groupby("name")['age'].transform('mean'),
        1
    )
)
```

### 왜 `transform()`?

```text
groupby()
→ 그룹별 평균 계산

transform()
→ 그 평균값을 원래 데이터의 각 행에 맞춰 반환
```

> ⭐ **핵심:**
> 전체 평균보다 **그룹의 특성을 반영한 평균값**으로 결측치를 채우는 방식.

---

## Embarked

최빈값으로 결측치를 채운다.

```python
train['embarked'] = train['embarked'].fillna(
    train['embarked'].mode().values[0]
)
```

> `mode()` → 가장 많이 등장하는 값
> `mode().values[0]` → 실제 값을 꺼냄

---

# 3-5. 불필요한 컬럼 삭제

```python
train = train.drop(
    columns=['ticket', 'cabin']
)
```

### 삭제 이유

* `ticket` → 직접적인 활용이 어려움
* `cabin` → 결측이 많고 그대로 사용하기 어려움

> ⭐ **모든 컬럼을 무조건 넣는 것이 좋은 것이 아니다.**
>
> **의미가 부족하거나 결측이 심한 변수는 제거하는 것도 전처리이다.**

---

# 3-6. 파생변수 생성

## `age_group`

나이를 10년 단위로 그룹화

```python
train["age_group"] = train["age"] // 10
```

예:

```text
23세 → 2
35세 → 3
47세 → 4
```

---

## `family_cnt`

전체 가족/동행 인원

```python
train['family_cnt'] = (
    train['sibsp']
    + train['parch']
    + 1
)
```

> `+1`은 **본인**을 포함하기 위해 사용.

---

## `fare_rate`

1인당 요금

```python
train['fare_rate'] = (
    train['fare']
    / train['family_cnt']
)
```

### 파생변수의 의미

```text
기존 데이터
sibsp + parch + fare
        ↓
파생변수
family_cnt + fare_rate
```

> ⭐ **핵심:**
> 원본 데이터에서 바로 보이지 않는 정보를 **새로운 Feature로 만들어 모델에 제공**한다.

---

# 3-7. Test의 Fare 결측치 처리

Test 데이터의 `fare` 결측치를 확인한 후

```python
family_cnt == 1
pclass == 3
```

조건에 해당하는 Train 데이터의 평균 요금을 사용한다.

```python
v = train[
    (train['family_cnt'] == 1)
    & (train['pclass'] == 3)
]['fare'].mean()

test['fare'] = test['fare'].fillna(v)
```

그리고 다시

```python
test['fare_rate'] = (
    test['fare']
    / test['family_cnt']
)
```

를 계산한다.

---

# 4. 공통 함수 + 기본 모델

실습에서는 모델 평가를 편하게 하기 위해
`myscore222()`라는 공통 함수를 만든다.

## 함수의 기본 흐름

```text
DataFrame
 ↓
X / y 분리
 ↓
train_test_split()
 ↓
모델 생성
 ↓
model.fit()
 ↓
model.predict()
 ↓
Accuracy / F1 / Precision / Recall
 ↓
Confusion Matrix
```

### 기본 분할

```python
train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1234,
    shuffle=True
)
```

> Train : Test = **7 : 3**

---

# 5. 모델 비교

실습에서 사용한 주요 분류 모델

### Logistic Regression

```python
LogisticRegression()
```

→ 선형 기반 분류 모델

### Random Forest

```python
RandomForestClassifier(
    n_estimators=100,
    random_state=1234
)
```

→ 여러 Decision Tree를 결합하는 **Bagging 계열 모델**

### XGBoost

```python
XGBClassifier()
```

→ 대표적인 **Boosting 모델**

### LightGBM

```python
LGBMClassifier()
```

→ 빠르고 효율적인 Boosting 계열 모델

> ⭐ **모델마다 데이터에 반응하는 방식이 다르므로 여러 모델을 비교해보는 것이 중요하다.**

---

# 6. 2차 EDA

## 상관분석

문자열 데이터는 상관계수를 계산할 수 없기 때문에
**인코딩 후 분석**한다.

### Pearson

```python
train.corr()
```

→ **선형적인 관계**를 확인

### Spearman

```python
train.corr(method='spearman')
```

→ **순위/단조 관계**를 확인

> ⭐ Pearson = 선형관계
> ⭐ Spearman = 순위관계

---

# 7. Target과 Feature 관계 확인

`survived`를 기준으로 여러 Feature를 비교한다.

```text
embarked
pclass
sex
sibsp
parch
age_group
```

실습에서 확인한 경향:

* **S에서 탑승한 사람**
* **3등급**
* **남성**
* **가족 없이 혼자 탑승한 사람**
* **20~30대**

등에서 생존하지 못한 경우가 상대적으로 많이 나타남.

> ⭐ **EDA 결과는 단순한 그래프가 아니라**
>
> **"어떤 Feature가 생존 여부와 관련이 있는가?"**
> 를 판단하기 위한 자료이다.

---

# 8. 이상치 확인

Boxplot을 이용해

```python
['age', 'sibsp', 'parch', 'fare']
```

의 이상치를 확인한다.

실습에서 확인한 대표적인 값:

```text
sibsp = 8
fare ≈ 500
```

→ 이상치 여부를 확인할 필요가 있음.

> ⭐ 이상치가 발견되었다고 **무조건 삭제하는 것은 아니다.**
>
> 실제 데이터에서 의미 있는 값일 수도 있으므로 원인을 먼저 확인한다.

---

# 9. 2차 전처리 & 파생변수

## `IsAlone`

혼자 탑승했는지 여부

```python
train['IsAlone'] = 1

train['IsAlone'].loc[
    train['family_cnt'] > 1
] = 0
```

```text
혼자 → 1
가족/동행 있음 → 0
```

---

## `FareBin`

요금을 4개의 구간으로 나눈다.

```python
train['FareBin'] = pd.qcut(
    train['fare'],
    4
)
```

이후 `LabelEncoder()`를 이용해 숫자로 변환한다.

```python
label = LabelEncoder()

train['FareBin'] = label.fit_transform(
    train['FareBin']
)
```

> ⭐ **연속적인 값을 구간화(Binning)해서 새로운 Feature로 활용**

---

# 10. 점수 변화 확인

실습에서 전처리 단계별 성능을 비교한다.

```text
1차 : 0.8097
      ↓
2차 : 0.8284
```

추가로 원핫인코딩을 적용하면서 성능을 비교한다.

> ⭐ 중요한 것은 **최종 점수 하나만 보는 것이 아니라**
>
> **"어떤 전처리를 추가했을 때 점수가 어떻게 변했는가?"**
>
> 를 비교하는 것.

---

# 11. Confusion Matrix

## 혼동행렬이란?

분류 모델의 **정답/오답을 세부적으로 분석하는 방법**

| 실제 \ 예측 |  0 |  1 |
| ------- | -: | -: |
| 0       | TN | FP |
| 1       | FN | TP |

### TN

실제 0 → 예측 0

### FP

실제 0 → 예측 1

### FN

실제 1 → 예측 0

### TP

실제 1 → 예측 1

---

# 12. 분류 평가 지표

## Accuracy

> 전체 데이터 중 **맞춘 비율**

```text
(TP + TN)
----------------
TP + TN + FP + FN
```

---

## Precision

> 모델이 **1이라고 예측한 것 중 실제 1인 비율**

```text
TP
---------
TP + FP
```

### 기억하기

> **"살았다고 했는데 그중 진짜 산 사람은?"**

→ FP가 적을수록 좋음

---

## Recall

> 실제 1인 데이터 중 모델이 **1이라고 찾아낸 비율**

```text
TP
---------
TP + FN
```

### 기억하기

> **"실제로 산 사람을 얼마나 놓치지 않았나?"**

→ FN이 적을수록 좋음

---

## F1 Score

> Precision과 Recall의 **조화평균**

```text
2 × Precision × Recall
-----------------------
Precision + Recall
```

또는

```text
2TP
----------------
2TP + FP + FN
```

### 핵심

> **Precision과 Recall 둘 다 중요할 때 사용하는 지표**

---

# 13. Titanic 실습의 오답 분석

실습 결과에서 확인한 값:

```text
TN = 144
TP = 76

FP = 22
FN = 26
```

### 정답

```text
144 + 76 = 220
```

### 오답

```text
22 + 26 = 48
```

### FP

```text
실제로는 죽었는데
모델은 살았다고 예측
```

### FN

```text
실제로는 살았는데
모델은 죽었다고 예측
```

> ⭐ **Confusion Matrix를 보면 단순히 "몇 % 맞췄다"를 넘어**
>
> **어떤 종류의 실수를 많이 하는지 알 수 있다.**

---

# 14. 교차검증(Cross Validation)

## 왜 사용하는가?

Train/Test를 한 번만 나눠서 평가하면
**특정 데이터 분할에 따라 점수가 달라질 수 있다.**

그래서 데이터를 여러 번 나눠 학습/검증한다.

---

# 15. KFold

```python
KFold(
    n_splits=5,
    shuffle=True,
    random_state=1234
)
```

### 의미

```text
전체 데이터
 ↓
5개로 분할
 ↓
1번: 4개 학습 + 1개 검증
2번: 4개 학습 + 1개 검증
3번: 4개 학습 + 1개 검증
4번: 4개 학습 + 1개 검증
5번: 4개 학습 + 1개 검증
 ↓
5번의 결과 평균
```

> ⭐ `n_splits=5`
> → 데이터를 5등분하여 **5번 학습/검증**

---

# 16. StratifiedKFold

KFold와 비슷하지만 **분류 문제에서 Target의 비율을 유지**한다.

```python
StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=1234
)
```

### 핵심 차이

```text
KFold
→ 단순하게 데이터 분할

StratifiedKFold
→ y의 클래스 비율을 고려하여 분할
```

> ⭐ **분류 문제에서는 StratifiedKFold가 특히 유용하다.**

---

# 17. cross_val_score()

KFold를 직접 작성하지 않고
교차검증을 간단하게 실행할 수 있다.

```python
tot_acc = cross_val_score(
    model,
    X,
    y,
    scoring="accuracy",
    cv=kf
)
```

결과:

```python
print(tot_acc)
print(np.mean(tot_acc))
```

→ 각 Fold의 점수와 평균 점수를 확인한다.

---

# 18. cross_validate()

여러 평가 지표를 동시에 확인할 수 있다.

```python
tot_dict = cross_validate(
    model,
    X,
    y,
    scoring=["accuracy", "f1"],
    cv=kf
)
```

```python
tot_acc = tot_dict['test_accuracy']
tot_f1 = tot_dict['test_f1']
```

> ⭐ `cross_val_score()` → 하나의 평가 지표
> ⭐ `cross_validate()` → 여러 평가 지표

---

# 19. GridSearchCV

## 목적

> **교차검증 + 하이퍼파라미터 튜닝**

### 하이퍼파라미터란?

모델이 학습하기 전에 사람이 설정하는 값.

Random Forest 예:

```text
n_estimators
min_samples_split
min_samples_leaf
max_depth
...
```

---

## 실습에서 탐색한 값

```python
myparam = {
    "n_estimators": [100, 200, 300],
    "min_samples_split": [2, 3, 4],
    "min_samples_leaf": [1, 2, 3]
}
```

총 여러 조합을 만들어 각각 비교한다.

```python
gcv_model = GridSearchCV(
    model,
    param_grid=myparam,
    scoring="f1",
    refit=True,
    cv=kf
)
```

### 결과

```python
gcv_model.best_params_
gcv_model.best_score_
```

→ 가장 좋은 파라미터와 그때의 점수를 확인한다.

> ⭐ `refit=True`
>
> → 가장 좋은 파라미터 조합으로 **최종 모델을 다시 학습해 둠**
>
> → 따라서 `best_params_`를 확인하고 모델을 다시 만들 필요가 없다.

### ⚠️ 주의

> **튜닝을 많이 한다고 무조건 좋은 것은 아니다.**
>
> 너무 특정 데이터에 맞추면 **Overfitting(과적합)**이 발생할 수 있다.

---

# 20. Ensemble : 모델 결합

## Voting

여러 모델의 예측을 결합해서 최종 예측을 결정한다.

실습에서는

```text
Logistic Regression
        +
Decision Tree
        +
KNN
        ↓
VotingClassifier
```

를 사용한다.

```python
vt_model = VotingClassifier(
    estimators=[
        ("lr", model1),
        ("dt", model2),
        ("knn", model3)
    ],
    voting="hard"
)
```

---

# 21. Bagging vs Boosting

| 구분   | Bagging         | Boosting          |
| ---- | --------------- | ----------------- |
| 핵심   | 같은 모델 여러 개      | 여러 모델을 순차적으로 결합   |
| 학습   | 병렬적             | 순차적               |
| 데이터  | Bootstrap 샘플링   | 이전 모델의 오답에 집중     |
| 목적   | **Variance 감소** | **Bias 감소**       |
| 대표 예 | Random Forest   | XGBoost, LightGBM |

### 쉽게 기억

```text
Bagging
→ 여러 명이 각자 문제를 풀고 결과를 합침

Boosting
→ 앞사람이 틀린 문제를 다음 사람이 집중해서 풂
```

---

# 22. 임계치(Threshold) 조정

모델의 `predict()` 결과만 사용하는 것이 아니라
`predict_proba()`를 이용해서 **1로 판단하는 기준을 직접 조절**할 수 있다.

```python
proba = model.predict_proba(X_test3)
proba_p = proba[:, 1]
```

예를 들어 기본적으로

```text
확률 >= 0.5 → 1
확률 < 0.5 → 0
```

처럼 판단하지만,

```python
Binarizer(threshold=0.1)
```

처럼 임계값을 바꿀 수도 있다.

### Threshold가 낮아지면?

```text
1로 판정하기 쉬워짐
        ↓
Recall ↑ 가능
        ↓
하지만 FP ↑ 가능
```

### Threshold가 높아지면?

```text
1로 판정하기 어려워짐
        ↓
Precision ↑ 가능
        ↓
하지만 FN ↑ 가능
```

> ⭐ **임계치는 평가 지표와 비즈니스 목적에 따라 조절할 수 있다.**

---

# ⭐ 최종 핵심 정리

## 이 실습에서 가장 중요한 흐름

```text
① 데이터 확인
   ↓
② 결측치 확인
   ↓
③ 문자열 → 숫자 인코딩
   ↓
④ 필요 없는 Feature 제거
   ↓
⑤ 파생 Feature 생성
   ↓
⑥ 모델 학습
   ↓
⑦ Accuracy / Precision / Recall / F1 평가
   ↓
⑧ EDA로 Feature 관계 확인
   ↓
⑨ 전처리 개선
   ↓
⑩ Confusion Matrix로 오답 분석
   ↓
⑪ Cross Validation으로 안정적인 성능 확인
   ↓
⑫ GridSearchCV로 최적 파라미터 탐색
   ↓
⑬ Ensemble로 여러 모델 결합
   ↓
⑭ Threshold 조정으로 예측 기준 변경
```

---

# 🔥 이번 실습에서 꼭 기억할 것

### 1. 전처리가 모델 성능에 큰 영향을 준다.

```text
원본 데이터
→ 결측치 처리
→ 인코딩
→ Feature 제거
→ Feature Engineering
→ One-Hot Encoding
```

**어떤 Feature를 어떻게 가공하느냐가 중요하다.**

---

### 2. 파생변수(Feature Engineering)가 중요하다.

```text
sibsp + parch + 1
→ family_cnt

fare / family_cnt
→ fare_rate

age // 10
→ age_group

family_cnt
→ IsAlone
```

> 기존 데이터에서 **문제 해결에 도움이 되는 새로운 정보를 만들어내는 과정**

---

### 3. Accuracy 하나만 보면 안 된다.

```text
Accuracy
Precision
Recall
F1
Confusion Matrix
```

각각 다른 관점에서 모델을 평가한다.

특히 **Precision과 Recall의 균형을 보고 싶다면 F1 Score**를 확인한다.

---

### 4. Confusion Matrix는 "오답의 종류"를 보여준다.

```text
FP → 실제 0인데 1이라고 예측
FN → 실제 1인데 0이라고 예측
```

> 단순히 "틀렸다"가 아니라
> **어떤 방향으로 틀렸는지**를 확인할 수 있다.

---

### 5. 교차검증은 모델 성능을 더 안정적으로 평가하기 위한 방법이다.

```text
KFold
→ 단순 분할

StratifiedKFold
→ 분류 문제에서 y 비율 유지
```

---

### 6. GridSearchCV는 "좋은 모델"을 찾는 것이 아니라

> **좋은 하이퍼파라미터 조합을 탐색하는 과정**

이다.

```text
모델
+
여러 파라미터 조합
+
교차검증
↓
best_params_
best_score_
```

---

### 7. 최종적으로는 "점수 올리기"보다

```text
데이터 이해
→ 적절한 전처리
→ Feature Engineering
→ 적절한 평가 지표 선택
→ 오답 분석
→ 검증
→ 튜닝
```

이라는 **머신러닝 문제 해결 과정 자체를 이해하는 것**이 핵심이다.
